# Fourier Neural Operator with Autoregressive Training

This notebook demonstrates how to train a Fourier Neural Operator (FNO) as a
**learned time-stepping operator** using autoregressive (AR) training.

Instead of learning the entire solution trajectory, we learn a single-step
evolution operator $\mathcal{G}_\theta$ that advances the state forward in time:

$$\mathbf{u}(\mathbf{x}, k+1) = \mathcal{G}_\theta[\mathbf{u}(\mathbf{x}, k)]$$

where:
- **$\mathbf{u}(\mathbf{x}, k)$**: System state at timestep $k$ (displacement and velocity)
- **$\mathcal{G}_\theta$**: FNO operator that learns the time evolution
- **$\mathbf{x}$**: Spatial coordinates

### Autoregressive Rollout

To generate long trajectories, we apply the operator recursively:
$$\mathbf{u}(\mathbf{x}, n) = \underbrace{\mathcal{G}_\theta \circ \mathcal{G}_\theta \circ \cdots \circ \mathcal{G}_\theta}_{n \text{ times}}[\mathbf{u}(\mathbf{x}, 0)]$$

### Training Strategy

We use a windowed training approach:
- Sample random time windows from trajectories
- Train on multi-step predictions to learn the evolution operator

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optax
from fno_models import create_fno1d_ar, FNO1D_AR
from jaxtyping import Array, Float, PRNGKeyArray, Scalar
from tqdm import tqdm
from utils import load_and_preprocess_data, ProgressPlotter, visualize_results

We define the loss function for the AR model. This loss is a bit more complex
because we want to randomly sample windows of each trajectory in a batch
and compute the loss over those windows.
We have to do this because the AR model cannot predict the entire trajectory in
one go, and also to avoid overfitting to the start of the trajectory.

In [ ]:
def loss_fn_ar(
    model: FNO1D_AR,
    batch: Array,
    n_steps: int = 10,
    n_windows: int = 32,  # Number of random windows per training step
    key: PRNGKeyArray = jax.random.PRNGKey(0),
) -> Scalar:
    """
    Compute loss for FNO1D_AR training.
    """

    # Calculate valid range for random starts
    total_steps = batch.shape[1]
    max_start = total_steps - n_steps

    # Generate random starting indices for windows
    def predict_single_batch(x: Float[Array, "T W C"], key: PRNGKeyArray):
        random_starts = jax.random.randint(
            key,
            (n_windows,),
            0,
            max_start + 1,
        )

        def compute_window_loss(start_idx):
            window = jax.lax.dynamic_slice_in_dim(
                x,
                start_idx,
                n_steps,
                axis=0,
            )  # Shape: (n_steps, W, C)

            y_pred = model(window[0:1])

            # Compute MSE loss for the window
            window_loss = jnp.mean((y_pred - window) ** 2)

            return window_loss

        window_losses = jax.vmap(compute_window_loss)(random_starts)
        return jnp.mean(window_losses)

    # Split keys for each batch element
    batch_size = batch.shape[0]
    keys = jax.random.split(key, batch_size)

    batch_losses = jax.vmap(predict_single_batch)(batch, keys)
    loss = jnp.mean(batch_losses)
    return loss

We define our training step and training loop.

In [ ]:
@eqx.filter_jit
def training_step_ar(
    model: FNO1D_AR,
    optimizer,
    opt_state,
    batch: Array,
    train_step_key: PRNGKeyArray,
):
    """Single training step for FNO1D_AR."""

    @eqx.filter_value_and_grad
    def compute_loss(model):
        return loss_fn_ar(
            model,
            batch,
            key=train_step_key,
        )

    loss_value, grads = compute_loss(model)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)

    return model, opt_state, loss_value

In [ ]:
def train_fno_ar_model(
    model: FNO1D_AR,
    train_dataloader,
    test_dataloader=None,
    n_epochs: int = 100,
    learning_rate: float = 5e-4,
    grad_clip_norm: float = 1.0,
    use_teacher_forcing: bool = True,
    teacher_forcing_decay: float = 0.95,
    visualize_every_n_epochs: int = 10,
    save_plots: bool = True,
    n_steps_test: int = 200,
    load_path: str | None = None,
):
    print("Starting training of FNO1D_AR model...")

    # Load pretrained model if path provided
    initial_epoch = 0
    if load_path is not None:
        from pathlib import Path

        load_path_obj = Path(load_path)
        if load_path_obj.exists():
            model = eqx.tree_deserialise_leaves(load_path, model)
            # Extract epoch number from filename pattern model_XXXXXX.eqx
            stem = load_path_obj.stem  # Gets filename without extension
            if (
                stem.startswith("fno_ar_model_") and len(stem) == 19
            ):  # fno_ar_model_ + 6 digits
                initial_epoch = int(stem[13:])  # Extract the 6-digit number
            print(
                f"Loaded pretrained model from {load_path}, "
                f"starting from epoch {initial_epoch}"
            )
        else:
            print(
                f"Warning: Load path {load_path} does not exist, starting from scratch"
            )

    # If epochs is 0, just return the loaded model without training
    if n_epochs == 0:
        # Optionally save the model even if no training occurred
        if load_path is not None:
            save_path = f"data/fno_ar_model_{initial_epoch:06d}.eqx"
            # Only save if it's a different path
            if str(load_path) != save_path:
                eqx.tree_serialise_leaves(save_path, model)
                print(f"Model copied to {save_path}")
        return model, np.array([]), None, initial_epoch

    # Initialize progress plotter
    plotter = None
    if save_plots and test_dataloader is not None:
        plotter = ProgressPlotter(
            output_dir="tmp_fno_ar",
            model_name="FNO1D_AR",
            framerate=5,
        )

    # Setup optimizer
    batches_per_epoch = len(train_dataloader)
    total_steps = n_epochs * batches_per_epoch

    schedule = optax.cosine_onecycle_schedule(
        transition_steps=total_steps,
        peak_value=learning_rate,
    )
    optimizer = optax.chain(
        optax.clip_by_global_norm(grad_clip_norm),
        optax.adamw(schedule),
    )
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    losses = np.full(n_epochs, np.nan)
    losses_list = []

    train_step_key = jax.random.PRNGKey(42)

    with tqdm(range(n_epochs), desc="Training FNO1D_AR") as pbar:
        for epoch in pbar:
            epoch_losses = []

            for batch in train_dataloader:
                train_step_key, epoch_key = jax.random.split(train_step_key)

                model, opt_state, loss_value = training_step_ar(
                    model,
                    optimizer,
                    opt_state,
                    batch,
                    train_step_key,
                )
                epoch_losses.append(loss_value)

            avg_loss = jnp.mean(jnp.array(epoch_losses))
            losses_list.append(avg_loss)
            losses[epoch] = avg_loss

            pbar.set_postfix(
                {
                    "Loss": f"{avg_loss:.6f}",
                }
            )

            # Visualize results
            if plotter is not None and (epoch + 1) % visualize_every_n_epochs == 0:
                # Extend model to predict more steps for visualization
                extended_model = eqx.tree_at(
                    lambda m: m.n_steps,
                    model,
                    n_steps_test - 1,  # Use n_steps_test - 1 for visualization
                )
                plotter(
                    extended_model,
                    test_dataloader,
                    losses,
                    show_plot=False,
                    show_loss_plot=False,
                )

    # Save the model after training
    total_epochs = initial_epoch + n_epochs
    save_path = f"data/fno_ar_model_{total_epochs:06d}.eqx"
    eqx.tree_serialise_leaves(save_path, model)
    print(f"Model saved to {save_path}")

    return model, np.array(losses_list), plotter, total_epochs

We set up the model, data, and training parameters, then train the model.

In [ ]:

hidden_channels = 32
n_modes = 101
n_layers = 4
n_epochs = 0  # Set to 0 to skip training and just load
batch_size = 25
learning_rate = 1e-3
grad_clip_norm = 1.0
n_steps = 10  # Number of AR prediction steps (a subset of the training steps)
n_steps_train = 100  # steps used during training
n_steps_test = 200  # steps used during testing

# Load data
(
    train_dataloader,
    val_dataloader,
    test_dataloader,
) = load_and_preprocess_data(
    "data/string_nonlin_100_Gaussian_16000Hz_1.0s.npy",
    batch_size=batch_size,
    n_steps_train=n_steps_train,
    n_steps_test=n_steps_test,
)

# Get sample to determine data shapes
sample = next(iter(train_dataloader))
print(f"Sample input shape: {sample.shape}")
print(f"Sample target shape: {sample.shape}")

# Model configuration
n_spatial_points = sample.shape[2]
input_channels = sample.shape[-1]
output_channels = sample.shape[-1]
n_prediction_steps = min(n_steps, sample.shape[1])  # Don't exceed available data

print("Model configuration:")
print(f"  Input channels: {input_channels}")
print(f"  Output channels: {output_channels}")
print(f"  Hidden channels: {hidden_channels}")
print(f"  Fourier modes: {n_modes}")
print(f"  Layers: {n_layers}")
print(f"  Prediction steps: {n_prediction_steps}")

We create the AR model with n_steps - 1 because this is the number of steps
the model will predict given an initial input.

In [ ]:

key = jax.random.PRNGKey(42)
fno_ar_model = create_fno1d_ar(
    input_channels=input_channels,
    hidden_channels=hidden_channels,
    n_modes=n_modes,
    n_steps=n_steps - 1,
    output_channels=output_channels,
    n_layers=n_layers,
    key=key,
)

Finally, we train the model.

In [ ]:

trained_model, training_losses, plotter, total_epochs = train_fno_ar_model(
    fno_ar_model,
    train_dataloader,
    test_dataloader=test_dataloader,
    n_epochs=n_epochs,
    learning_rate=learning_rate,
    grad_clip_norm=grad_clip_norm,
    visualize_every_n_epochs=10,
    save_plots=True,
    n_steps_test=n_steps_test,
    load_path="data/fno_ar_model_005000.eqx",
)

if training_losses.size > 0:
    print(f"Training completed! Final loss: {training_losses[-1]:.6f}")
else:
    print("No training performed (n_epochs=0). Using loaded model.")

In [ ]:
# Predict a longer horizon
final_trained_model = eqx.tree_at(
    lambda m: m.n_steps,
    trained_model,
    n_steps_test - 1,
)

Visualise the results and extrapolation in time.

In [ ]:

visualize_results(
    final_trained_model,
    test_dataloader,
    training_losses,
    model_name="FNO1D_AR",
    show_loss_plot=False,
    n_steps_train=n_steps_train,
)

# Generate animation from training frames
if plotter is not None:
    plotter.render_animation("fno_ar_training.webm")